# Phase 2: Hybrid Weapon Detection Training (v4 - Google Colab T4)
This notebook is optimized for **Google Colab T4 GPU**. It implements the Hybrid YOLOv11-Swin Transformer architecture with 'Red Alert' false-positive reduction logic and persistent checkpointing to Google Drive.

## Step 1: Google Drive & Environment Setup
Mount Drive and install dependencies. **Note**: If you restart the runtime, you must run this cell and Step 2 again.

In [ ]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# 2. Install dependencies
!pip install ultralytics albumentations

# 3. Force compatible Numpy version (< 2.0)
!pip uninstall -y numpy && pip install "numpy<2.0"

import numpy as np
import torch
print(f"\n✅ Numpy Version: {np.__version__}")
print(f"✅ Torch Version: {torch.__version__}")
print(f"✅ GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE'}")

if np.version.version.startswith('2.'):
    print("\n[IMPORTANT] Numpy 2.x detected. Please click 'RESTART SESSION' in the Colab popup, then skip this cell and go to Step 2!")

## Step 2: Project Alignment & Core Imports

In [ ]:
import os
import sys
from pathlib import Path

# --- 1. Set Project Path ---
PROJECT_ROOT = Path("/content/drive/MyDrive/Real-Time-Weapon-Detection-Context-Aware-Red-Alert-System-v2/Real-Time-Weapon-Detection-Context-Aware-Red-Alert-System/")

if PROJECT_ROOT.exists():
    os.chdir(str(PROJECT_ROOT))
    if str(PROJECT_ROOT) not in sys.path:
        sys.path.append(str(PROJECT_ROOT))
    print(f"✅ Project Root Aligned: {PROJECT_ROOT}")
else:
    raise FileNotFoundError(f"Could not locate the project directory at {PROJECT_ROOT}")

# --- 2. Verify and Import ---
from models.hybrid_model import HybridWeaponDetector
from ultralytics.data.dataset import YOLODataset
from ultralytics.data.utils import check_det_dataset
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torch.cuda.amp import autocast, GradScaler
from tqdm import tqdm
import yaml

print("✅ Core modules and dependencies imported successfully!")

## Step 3: Dataset Cache Cleaning & Verification
**Run this cell** to fix 'No images found' errors by deleting old cache files and verifying Google Drive sync.

In [ ]:
def clean_dataset_caches(dataset_root):
    dataset_path = Path(dataset_root)
    for split in ['train', 'val', 'test']:
        split_dir = dataset_path / split
        cache_file = split_dir / "labels.cache"
        if cache_file.exists():
            print(f"🗑️ Removing old cache: {cache_file}")
            os.remove(cache_file)
        
        img_dir = split_dir / "images"
        if img_dir.exists():
            imgs = [f for f in os.listdir(img_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
            print(f"✅ {split.upper()}: Found {len(imgs)} images.")
            if len(imgs) == 0:
                print(f"  ⚠️ WARNING: {split} folder appears empty. Wait for GDrive sync!")
        else:
            print(f"  ❌ ERROR: {img_dir} does not exist.")

DATASET_ROOT = "data/processed/yolo_dataset"
clean_dataset_caches(DATASET_ROOT)

## Step 4: Hybrid Trainer & Hardware Optimization

In [ ]:
def get_dataloaders(data_yaml_path, batch_size=16, imgsz=640):
    data_cfg = check_det_dataset(data_yaml_path)
    train_set = YOLODataset(img_path=data_cfg['train'], imgsz=imgsz, augment=True, batch_size=batch_size, task='detect', data=data_cfg)
    val_set = YOLODataset(img_path=data_cfg['val'], imgsz=imgsz, augment=False, batch_size=batch_size, task='detect', data=data_cfg)
    
    train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True, collate_fn=train_set.collate_fn)
    val_loader = DataLoader(val_set, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True, collate_fn=val_set.collate_fn)
    
    return train_loader, val_loader

class HybridTrainer:
    def __init__(self, model, train_loader, val_loader, device="cuda", weights_dir="models/weights"):
        self.device = device
        self.model = model.to(device)
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.scaler = GradScaler()
        self.weights_dir = Path(weights_dir)
        self.weights_dir.mkdir(parents=True, exist_ok=True)
        
        self.criterion = model.head.compute_loss
        self.best_loss = float('inf')

    def save_checkpoint(self, optimizer, epoch, is_best=False, is_periodic=False):
        state = {
            'epoch': epoch,
            'model_state_dict': self.model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'best_loss': self.best_loss
        }
        torch.save(state, self.weights_dir / "last.pt")
        if is_best:
            torch.save(self.model.state_dict(), self.weights_dir / "best.pt")
        if is_periodic:
            torch.save(self.model.state_dict(), self.weights_dir / f"epoch_{epoch}.pt")

    def load_checkpoint(self):
        checkpoint_path = self.weights_dir / "last.pt"
        if checkpoint_path.exists():
            print(f"✅ Resuming from checkpoint: {checkpoint_path}")
            state = torch.load(checkpoint_path, map_location=self.device)
            self.model.load_state_dict(state['model_state_dict'])
            self.best_loss = state.get('best_loss', float('inf'))
            return state['epoch'], state['optimizer_state_dict']
        return 0, None

    def train_epoch(self, optimizer, epoch):
        self.model.train()
        pbar = tqdm(self.train_loader, desc=f"Epoch {epoch}")
        total_loss = 0
        for batch in pbar:
            imgs = batch['img'].to(self.device).float() / 255.0
            optimizer.zero_grad()
            with autocast():
                preds = self.model(imgs)
                loss = self.criterion(preds, batch, self.device)
            self.scaler.scale(loss).backward()
            self.scaler.step(optimizer)
            self.scaler.update()
            total_loss += loss.item()
            pbar.set_postfix({"loss": f"{loss.item():.4f}"})
        return total_loss / len(self.train_loader)

    def run(self, epochs=50):
        start_epoch, opt_state = self.load_checkpoint()
        if start_epoch < 10:
            print("[INFO] Frozen Backbone Phase (Epochs 1-10)")
            self.model.backbone.freeze()
            lr = 1e-4
        else:
            print("[INFO] Full Fine-tuning Phase")
            self.model.backbone.unfreeze()
            lr = 1e-5
        optimizer = optim.AdamW(filter(lambda p: p.requires_grad, self.model.parameters()), lr=lr)
        if opt_state: optimizer.load_state_dict(opt_state)
        for epoch in range(start_epoch + 1, epochs + 1):
            if epoch == 10:
                print("\n[INFO] Milestone Reached: Unfreezing Backbone")
                self.model.backbone.unfreeze()
                optimizer = optim.AdamW(self.model.parameters(), lr=1e-5)
            avg_loss = self.train_epoch(optimizer, epoch)
            is_best = avg_loss < self.best_loss
            if is_best: self.best_loss = avg_loss
            self.save_checkpoint(optimizer, epoch, is_best=is_best, is_periodic=(epoch % 5 == 0))
            print(f"Epoch {epoch} complete. Avg Loss: {avg_loss:.4f} | Best: {self.best_loss:.4f}")

## Step 5: Initiation & Training Execution

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model = HybridWeaponDetector(backbone_variant="yolo11m.pt", nc=3, device=device)
model.head.alpha = torch.tensor([1.0, 1.0, 2.5], device=device)

DATA_YAML_PATH = "data/processed/yolo_dataset/data.yaml"
if os.path.exists(DATA_YAML_PATH):
    train_loader, val_loader = get_dataloaders(DATA_YAML_PATH, batch_size=16)
    trainer = HybridTrainer(model, train_loader, val_loader, device=device)
    trainer.run(epochs=50)
else:
    print(f"❌ [ERROR] data.yaml not found at {DATA_YAML_PATH}.")